## QAT Test-Set Inference using PyTorch (Multi-Seed)

Evaluate QAT checkpoints on the **test set** (ImageNet val split).

- **INT8**: `checkpoints/qat/int8_in{8,4,2,1}b/seed_{1,2,42}/`
- **INT4**: `checkpoints/qat/int4_in{8,4,2,1}b/seed_{1,2,42}/`

Only runs for which training has completed (checkpoint exists) are evaluated.

Results saved to `resultsv2/test_runs/`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────
SPLIT = "val"
TEST_IMAGENET = "/home/pf4636/imagenet2"
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test/qat"
SKIP_EXISTING = True

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [3]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from src.eval import evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [ ]:
QAT_CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QAT_CONFIGS = [
    {"qat_precision": "int8", "input_bits": [8, 4, 2, 1]},
    {"qat_precision": "int4", "input_bits": [8, 4, 2, 1]},
]

checkpoints = {}
skipped = []
for qcfg in QAT_CONFIGS:
    prec = qcfg["qat_precision"]
    for bits in qcfg["input_bits"]:
        run_name = f"{prec}_in{bits}b"
        run_dir = QAT_CKPT_ROOT / run_name
        if not run_dir.exists():
            skipped.append(f"{prec}/in{bits}b (no dir)")
            continue
        for seed_dir in sorted(run_dir.iterdir()):
            if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
                continue
            weights = seed_dir / "qat_modelopt_best.pth"
            mostate = seed_dir / "qat_modelopt_best_mostate.pt"
            seed_num = int(seed_dir.name.split("_")[1])
            fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / seed_dir.name / "best.pth"
            if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
                skipped.append(f"{prec}/in{bits}b/{seed_dir.name}")
                continue
            checkpoints[(prec, bits, seed_dir.name)] = {
                "weights": weights,
                "mostate": mostate,
                "fp32_ckpt": fp32_ckpt,
                "seed": seed_num,
            }

print(f"QAT checkpoints found: {len(checkpoints)}")
for (prec, bits, seed), paths in checkpoints.items():
    print(f"  {prec} / in{bits}b / {seed}")
if skipped:
    print(f"\nSkipped (not complete): {', '.join(skipped)}")
print(f"Test set: {TEST_IMAGENET}")

In [ ]:
def load_qat_model(prec: str, bits: int, seed_name: str) -> nn.Module:
    paths = checkpoints[(prec, bits, seed_name)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_IMAGENET)
    dataset = build_imagenet_dataset(test_cfg, split=SPLIT)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## QAT Evaluation (All Seeds)

In [ ]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
criterion = nn.CrossEntropyLoss()

for (prec, bits, seed_name), paths in checkpoints.items():
    seed_num = paths["seed"]
    run_id = f"torch_qat_{prec}_b{bits}_cuda"
    result_path = OUT_DIR / seed_name / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  {prec}/in{bits}b/{seed_name}: exists, skipping")
        with open(result_path) as f:
            all_records.append(json.load(f))
        continue

    print(f"\n--- QAT {prec.upper()}, input_bits={bits}, {seed_name} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=seed_num,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model(prec, bits, seed_name)
    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, test_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": prec,
            "input_quant_bits": bits,
            "seed": seed_num,
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    all_records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(all_records)} runs complete.")

## Per-Run Results

In [ ]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    prec = extra.get("qat_precision", "int8")
    seed = extra.get("seed", 42)
    rows.append({
        "qat_precision": f"qat_{prec}",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["qat_precision", "input_bits", "seed"], ascending=[True, True, True]
).reset_index(drop=True)
df_all

## Averaged Results (mean ± std across seeds)

In [ ]:
avg_df = df_all.groupby(["qat_precision", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["qat_precision", "input_bits"], ascending=[True, True]
).reset_index(drop=True)
avg_df

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_torch_avg_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")